# Point-in-time difference: the same SEC figure, two ways

This is a data-integrity demonstration, not a strategy result. It makes no performance claim, no signal claim, and is not a backtest. The figures shown are as filed with the SEC through public EDGAR data.

The value that stands today is the latest filing for the reporting period. The historical view is the value that was on file as of a chosen historical date, using a filing-date cutoff, never the reporting period. The free client reads live SEC EDGAR and reflects what SEC currently serves.

In [ ]:
import os

os.environ.setdefault("SEC_EDGAR_USER_AGENT", "arkleon point-in-time example (founder@arkleon.com)")
from arkleon import EdgarClient

client = EdgarClient()

def latest_consolidated(factset, period_end):
    xs = [f for f in factset if f.period_end == period_end
          and f.segments is None and f.coreg is None]
    xs.sort(key=lambda f: f.filed)
    return xs[-1]


In [ ]:
SAMPLE = [
    {"cik": 4447, "tag": "ProceedsFromPaymentsForOtherFinancingActivities",
     "period_end": "2015-06-30", "as_of": "2015-08-07"},
    {"cik": 3570, "tag": "OtherNoncashIncomeExpense",
     "period_end": "2020-12-31", "as_of": "2022-02-24"},
]

In [ ]:
import pandas as pd

results = []
for row in SAMPLE:
    fs = client.concept(row["cik"], row["tag"], taxonomy="us-gaap")
    today = latest_consolidated(fs, row["period_end"])
    asof = latest_consolidated(fs.as_of(row["as_of"]), row["period_end"])
    results.append({
        "cik": row["cik"],
        "tag": row["tag"],
        "reporting_period": row["period_end"],
        "as_of_date": row["as_of"],
        "as_of_value": asof.value,
        "as_of_filed": asof.filed,
        "today_value": today.value,
        "today_filed": today.filed,
        "differs": today.value != asof.value,
        "abs_difference": abs(today.value - asof.value),
    })

df = pd.DataFrame(results, columns=[
    "cik", "tag", "reporting_period", "as_of_date",
    "as_of_value", "as_of_filed", "today_value", "today_filed",
    "differs", "abs_difference",
])
df

In [ ]:
n_differ = sum(r["differs"] for r in results)
print(f"Sampled figures that differ: {n_differ} of {len(results)}")
import json; print("RESULTS_JSON=" + json.dumps(results, sort_keys=True))

The figures are as filed with the SEC. The free client's point-in-time view is best-effort and reflects only what SEC currently serves. No corpus revision count is asserted. No performance or signal claim is made.

## Optional: the certified /v1 API path

With an `ARKLEON_API_KEY` set, the same question can be asked against the certified `https://arkleon.com/v1` service.

In [ ]:
if os.environ.get("ARKLEON_API_KEY"):
    # The paid DataClient/pit_facts path can answer the same question.
    print("ARKLEON_API_KEY set; the optional /v1 path is available.")
else:
    print("No ARKLEON_API_KEY set; skipping the /v1 path.")